In [1]:
from pathlib import Path
import os

REPO_ROOT = Path.cwd().parents[1]

DATA_PATH = REPO_ROOT / "data" / "all_heuristics_dataset.pkl"   # preferred
# DATA_PATH = REPO_ROOT / "data" / "all_heuristics_dataset.csv" # fallback

OUT_DIR = REPO_ROOT / "notebooks" / "Emirhan" / "outputs" / "notebook_test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists(), DATA_PATH)
print("OUT_DIR:", OUT_DIR)


REPO_ROOT: /Users/emirhangunes/VSCode/ppp-performance-prediction
DATA_PATH exists: True /Users/emirhangunes/VSCode/ppp-performance-prediction/data/all_heuristics_dataset.pkl
OUT_DIR: /Users/emirhangunes/VSCode/ppp-performance-prediction/notebooks/Emirhan/outputs/notebook_test


In [2]:
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(sys.path[0])


/Users/emirhangunes/VSCode/ppp-performance-prediction


In [3]:
from src.api.mock_client import MockLLMClient
from src.api.parse import parse_cap_response, parse_ppp_response
from src.pipelines.schemas import LLMResultRow

print("Imports OK")


Imports OK


In [4]:
import pandas as pd

df = pd.read_pickle(DATA_PATH) if DATA_PATH.suffix == ".pkl" else pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head(3)


Rows: 15507
Columns: ['heuristic_id', 'raw_app_type', 'instance_scale', 'filename', 'strategy', 'algorithm', 'code', 'objective', 'task_name', 'parse_ok', 'is_timeout']


,heuristic_id,raw_app_type,instance_scale,filename,strategy,algorithm,code,objective,task_name,parse_ok,is_timeout
0,pop_0_op_e1_n0_251224_134701,bin_greedy,results_20251224_134317_bp_online_qwen25coder3...,program_pop_0_op_e1_n0_251224_134701.json,e1,The new algorithm assigns scores based on a co...,"import numpy as np\n\ndef score(item, bins):\n...",1.51534,BinPacking,True,False
1,pop_0_op_e1_n10_251224_134701,bin_greedy,results_20251224_134317_bp_online_qwen25coder3...,program_pop_0_op_e1_n10_251224_134701.json,e1,The new algorithm calculates scores for each b...,"import numpy as np\n\ndef score(item, bins):\n...",0.32770,BinPacking,True,False
2,pop_0_op_e1_n11_251224_134701,bin_greedy,results_20251224_134317_bp_online_qwen25coder3...,program_pop_0_op_e1_n11_251224_134701.json,e1,The new algorithm calculates scores for each b...,"import numpy as np\n\ndef score(item, bins):\n...",1.51534,BinPacking,True,False


In [5]:
# robust bool handling (in case parse_ok is string)
def to_bool(x):
    if isinstance(x, bool): return x
    if isinstance(x, str): return x.strip().lower() == "true"
    return False

df["parse_ok_bool"] = df["parse_ok"].apply(to_bool)
df["is_timeout_bool"] = df["is_timeout"].apply(to_bool)

df_f = df[(df["parse_ok_bool"] == True) & (df["is_timeout_bool"] == False)].copy()

print("Filtered rows:", len(df_f))
df_f.head(3)[["heuristic_id","raw_app_type","strategy","objective","parse_ok","is_timeout"]]


Filtered rows: 15507


,heuristic_id,raw_app_type,strategy,objective,parse_ok,is_timeout
0,pop_0_op_e1_n0_251224_134701,bin_greedy,e1,1.51534,True,False
1,pop_0_op_e1_n10_251224_134701,bin_greedy,e1,0.32770,True,False
2,pop_0_op_e1_n11_251224_134701,bin_greedy,e1,1.51534,True,False


In [6]:
cap_template = (REPO_ROOT / "src" / "prompt_templates" / "cap.md").read_text(encoding="utf-8")
ppp_template = (REPO_ROOT / "src" / "prompt_templates" / "ppp_with_refs.md").read_text(encoding="utf-8")

print("CAP template length:", len(cap_template))
print("PPP template length:", len(ppp_template))


CAP template length: 753
PPP template length: 787


In [7]:
client = MockLLMClient()

sample = df_f.iloc[0]
cap_prompt = cap_template.format(code=sample["code"])

resp = client.generate(cap_prompt, temperature=0.2, max_tokens=256, stop=None, meta={"stage": "cap"})
core_idea, conf, ok, err, extra = parse_cap_response(resp.text)

print("CAP parse_ok:", ok)
print("CAP error:", err)
print("CAP core_idea:", core_idea)
print("LLMResponse meta:", resp.model, resp.prompt_hash, resp.latency_s, resp.cached)


CAP parse_ok: True
CAP error: None
CAP core_idea: This heuristic scores candidate actions using remaining capacity and penalty/bonus terms to prioritize placements that reduce waste and balance utilization.
LLMResponse meta: mock d96961466efcd57742a930f1e27d362ac0ba3f14b3ac76734f11dd42cde28e70 5.0067901611328125e-06 False


In [9]:
import csv

cap_out = OUT_DIR / "cap_test.csv"

cap_rows = []
for _, row in df_f.head(10).iterrows():
    prompt = cap_template.format(code=row["code"])
    resp = client.generate(prompt, temperature=0.2, max_tokens=256, stop=None, meta={"stage":"cap", "heuristic_id": row["heuristic_id"]})
    core_idea, conf, ok, err, extra = parse_cap_response(resp.text)

    cap_rows.append(LLMResultRow(
        heuristic_id=str(row["heuristic_id"]),
        raw_app_type=str(row["raw_app_type"]),
        strategy=str(row["strategy"]) if pd.notna(row["strategy"]) else None,
        objective=float(row["objective"]) if pd.notna(row["objective"]) else None,
        llm_value=core_idea,                 # CAP -> string
        llm_confidence=conf,
        parse_ok=bool(ok),
        parse_error=err,
        model=resp.model,
        prompt_hash=resp.prompt_hash,
        latency_s=resp.latency_s,
        cached=resp.cached,
        raw_text=resp.text,                  # keep raw for debugging in notebook
    ))

# write CSV
with open(cap_out, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=LLMResultRow.columns())
    w.writeheader()
    for r in cap_rows:
        w.writerow(r.to_dict())

print("Wrote:", cap_out)


Wrote: /Users/emirhangunes/VSCode/ppp-performance-prediction/notebooks/Emirhan/outputs/notebook_test/cap_test.csv


In [10]:
cap_map = {r.heuristic_id: r.llm_value for r in cap_rows if r.parse_ok and isinstance(r.llm_value, str)}
print("CAP map size:", len(cap_map))
list(cap_map.items())[:2]


CAP map size: 10


[('pop_0_op_e1_n0_251224_134701',
  'This heuristic scores candidate actions using remaining capacity and penalty/bonus terms to prioritize placements that reduce waste and balance utilization.'),
 ('pop_0_op_e1_n10_251224_134701',
  'This heuristic scores candidate actions using remaining capacity and penalty/bonus terms to prioritize placements that reduce waste and balance utilization.')]

In [11]:
# pick an app type with enough rows in df_f
app_type = df_f["raw_app_type"].value_counts().index[0]
df_app = df_f[df_f["raw_app_type"] == app_type].copy()

df_app = df_app[pd.notna(df_app["objective"])].sort_values("objective")  # lower is better
print("Using app_type:", app_type, "rows:", len(df_app))

best = df_app.iloc[0]
mid  = df_app.iloc[len(df_app)//2]
worst = df_app.iloc[-1]

refs = [best, mid, worst]
[(r["heuristic_id"], r["objective"]) for r in refs]


Using app_type: puzzle_astar rows: 4858


[('pop_19_op_e2_n16_251024_144718', np.float64(0.4574)),
 ('pop_3_op_m1_n19_251005_004701', np.float64(2.01919)),
 ('pop_1_op_e1_n10_251018_094429', np.float64(3.65888))]

In [12]:
def build_references_block(ref_rows):
    lines = []
    for i, rr in enumerate(ref_rows, start=1):
        hid = str(rr["heuristic_id"])
        core = cap_map.get(hid, f"(missing CAP core_idea for {hid})")
        obj = float(rr["objective"])
        lines.append(f"{i})")
        lines.append("Core Idea:")
        lines.append(core)
        lines.append(f"Objective: {obj}")
        lines.append("")
    return "\n".join(lines).strip() + "\n"

# choose a target from our CAP subset (so target_core exists)
target_row = df_f.head(10).iloc[0]
target_id = str(target_row["heuristic_id"])
target_core = cap_map.get(target_id)

print("Target:", target_id)
print("Target core exists:", target_core is not None)

references_block = build_references_block(refs)

ppp_prompt = ppp_template.format(
    raw_app_type=str(app_type),
    references_block=references_block,
    target_core=target_core if target_core else "(missing CAP core idea)",
)

resp = client.generate(ppp_prompt, temperature=0.2, max_tokens=256, stop=None, meta={"stage":"ppp", "heuristic_id": target_id})
pred, conf, ok, err, extra = parse_ppp_response(resp.text)

print("PPP parse_ok:", ok)
print("PPP error:", err)
print("PPP prediction:", pred)
print("PPP confidence:", conf)
print("PPP justification (if any):", extra.get("justification"))


Target: pop_0_op_e1_n0_251224_134701
Target core exists: True
PPP parse_ok: True
PPP error: None
PPP prediction: 0.42
PPP confidence: 0.15
PPP justification (if any): The target core idea resembles the better reference heuristic more than the weaker one, so the predicted objective is closer to the low objective range.


In [13]:
import csv
import pandas as pd

ppp_out = OUT_DIR / "ppp_test.csv"

targets = df_f.head(5).copy()

ppp_rows = []
for _, t in targets.iterrows():
    tid = str(t["heuristic_id"])
    tcore = cap_map.get(tid)
    if not tcore:
        continue

    # choose refs from same app type as target
    app = str(t["raw_app_type"])
    df_app = df_f[df_f["raw_app_type"] == app].copy()
    df_app = df_app[pd.notna(df_app["objective"])].sort_values("objective")
    if len(df_app) < 3:
        continue
    ref_rows = [df_app.iloc[0], df_app.iloc[len(df_app)//2], df_app.iloc[-1]]
    references_block = build_references_block(ref_rows)

    prompt = ppp_template.format(
        raw_app_type=app,
        references_block=references_block,
        target_core=tcore,
    )

    resp = client.generate(prompt, temperature=0.2, max_tokens=256, stop=None, meta={"stage":"ppp", "heuristic_id": tid})
    pred, conf, ok, err, extra = parse_ppp_response(resp.text)

    ppp_rows.append(LLMResultRow(
        heuristic_id=tid,
        raw_app_type=app,
        strategy=str(t["strategy"]) if pd.notna(t["strategy"]) else None,
        objective=float(t["objective"]) if pd.notna(t["objective"]) else None,
        llm_value=pred,                   # PPP -> float prediction
        llm_confidence=conf,
        parse_ok=bool(ok),
        parse_error=err,
        model=resp.model,
        prompt_hash=resp.prompt_hash,
        latency_s=resp.latency_s,
        cached=resp.cached,
        raw_text=resp.text,
    ))

with open(ppp_out, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=LLMResultRow.columns())
    w.writeheader()
    for r in ppp_rows:
        w.writerow(r.to_dict())

print("Wrote:", ppp_out, "rows:", len(ppp_rows))


Wrote: /Users/emirhangunes/VSCode/ppp-performance-prediction/notebooks/Emirhan/outputs/notebook_test/ppp_test.csv rows: 5


In [14]:
cap_df = pd.read_csv(cap_out)
ppp_df = pd.read_csv(ppp_out)

print("CAP CSV preview:")
display(cap_df.head(3))

print("PPP CSV preview:")
display(ppp_df.head(3))

print("PPP predicted values:", ppp_df["llm_value"].head().tolist())


CAP CSV preview:


,heuristic_id,raw_app_type,strategy,objective,llm_value,llm_confidence,parse_ok,parse_error,model,prompt_hash,latency_s,cached,raw_text
0,pop_0_op_e1_n0_251224_134701,bin_greedy,e1,1.51534,This heuristic scores candidate actions using ...,NaN,True,NaN,mock,d96961466efcd57742a930f1e27d362ac0ba3f14b3ac76...,0.000018,False,"CAP result:\n{\n ""core_idea"": ""This heuristic..."
1,pop_0_op_e1_n10_251224_134701,bin_greedy,e1,0.32770,This heuristic scores candidate actions using ...,NaN,True,NaN,mock,4acda464aafd3beca83c65fb0788f11ea6e66c1fbfec36...,0.000002,False,"CAP result:\n{\n ""core_idea"": ""This heuristic..."
2,pop_0_op_e1_n11_251224_134701,bin_greedy,e1,1.51534,This heuristic scores candidate actions using ...,NaN,True,NaN,mock,6ccaada4e2dc06b72c30a6b4bafa1bf8e91d307890c65e...,0.000002,False,"CAP result:\n{\n ""core_idea"": ""This heuristic..."


PPP CSV preview:


,heuristic_id,raw_app_type,strategy,objective,llm_value,llm_confidence,parse_ok,parse_error,model,prompt_hash,latency_s,cached,raw_text
0,pop_0_op_e1_n0_251224_134701,bin_greedy,e1,1.51534,0.42,0.15,True,NaN,mock,507c82662e576d69d59331d8eb00531a7f8f8f713356d7...,0.000008,False,"PPP prediction result:\n{\n ""prediction"": 0.4..."
1,pop_0_op_e1_n10_251224_134701,bin_greedy,e1,0.32770,0.42,0.15,True,NaN,mock,507c82662e576d69d59331d8eb00531a7f8f8f713356d7...,0.000008,False,"PPP prediction result:\n{\n ""prediction"": 0.4..."
2,pop_0_op_e1_n11_251224_134701,bin_greedy,e1,1.51534,0.42,0.15,True,NaN,mock,507c82662e576d69d59331d8eb00531a7f8f8f713356d7...,0.000003,False,"PPP prediction result:\n{\n ""prediction"": 0.4..."


PPP predicted values: [0.42, 0.42, 0.42, 0.42, 0.42]
